# 02 · Load every booking file, check every column, write one table

**Produces:** `data/clean/bookings_all.csv.gz` (one row per booking-test, both systems, same columns), `data/clean/rejects_bookings.csv` (rows we could not accept, with the reason) and `data/clean/column_checks.csv` (the PASS / WARN / FAIL table).

**Needs:** notebook 01 run, and the data pack in `data/raw/`.

**Two ideas in this notebook.**
1. The pack is about 110 booking files in two shapes. We write one small reader for each shape and loop over the files.
2. Before any cleaning we check every important column against what it should contain, on the data exactly as it arrived. Whatever fails is kept and counted, so *input = accepted + rejected*.

## CONFIG
Everything that depends on the machine or the period lives in this one cell, so nothing else needs editing.

In [ ]:
# ---------- imports ----------
import gzip
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# show more columns when we print a dataframe
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

# ---------- paths ----------
# the notebooks live in the notebooks folder, so the repo root is one folder up
if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

RAW = ROOT / "data" / "raw"       # the data pack as we received it (we never edit this)
CLEAN = ROOT / "data" / "clean"   # the files our notebooks write
SLIDES = ROOT / "slides"          # charts and the excel file for the slides
CLEAN.mkdir(parents=True, exist_ok=True)
SLIDES.mkdir(parents=True, exist_ok=True)

# ---------- the two periods we compare ----------
H1_2025 = ("2025-01-01", "2025-06-30 23:59:59")   # 23:59:59 so that 30 June is included
H1_2026 = ("2026-01-01", "2026-06-30 23:59:59")

# a small helper: prints PASS or FAIL but never stops the notebook
def check(name, condition, detail=""):
    if condition:
        print("PASS", name, detail)
    else:
        print("FAIL", name, detail)

print("repo root:", ROOT)

## 1 · MAP: two systems, one table
```
legacy/*.csv.gz   (one row per TEST)      ─┐
                                            ├─ loop over files ─ stack ─ column checks ─ one table
nova/*.jsonl.gz   (one object per BOOKING) ─┘
```
The nova tests are nested inside each booking, so we flatten them to one row per test.

In [ ]:
# ---------- reader for a legacy file ----------
def read_legacy_file(path):
    # legacy files are latin-1 and the vendor repeated the header line inside some of them
    df = pd.read_csv(path, encoding="latin-1", dtype=str)
    df = df[df["booking_id"] != "booking_id"]      # drop the repeated header lines
    df["system"] = "legacy"
    df["source_file"] = path.name
    return df


# ---------- reader for a nova file ----------
def read_nova_file(path):
    # nova writes one json object per booking, with the tests inside it
    rows = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            tests = r["tests"]
            if len(tests) == 0:
                # a booking with no tests: keep one empty row so we can see it and count it
                tests = [{"code": None, "price": None}]
            for t in tests:
                rows.append({
                    "booking_id": r["booking_id"],
                    "created_at": r["created_at"],
                    "slot_at": r["slot_at"],
                    "city": r["city"],
                    "clinic_id": r["clinic_id"],
                    "channel": r["channel"].lower(),     # nova writes CLINIC / HOME in capitals
                    "source": r["source"].lower(),
                    "patient_id": r["patient"]["patient_id"],
                    "test_code": t["code"],
                    "test_price": t["price"],
                    "home_fee": r["home_fee"],
                    "rescheduled_from": r["rescheduled_from"],
                    "status_at_export": r["status_at_export"],
                })
    df = pd.DataFrame(rows)
    df["system"] = "nova"
    df["source_file"] = path.name
    return df

### Do: test each reader on one file and time it

In [ ]:
legacy_files = sorted((RAW / "bookings" / "legacy").rglob("*.csv.gz"))
nova_files = sorted((RAW / "bookings" / "nova").rglob("*.jsonl.gz"))
print(len(legacy_files), "legacy files,", len(nova_files), "nova files")

t0 = time.time()
df_test_leg = read_legacy_file(legacy_files[0])
print("legacy file:", df_test_leg.shape, "in", round(time.time() - t0, 2), "seconds")

t0 = time.time()
df_test_nova = read_nova_file(nova_files[0])
print("nova file  :", df_test_nova.shape, "in", round(time.time() - t0, 2), "seconds")

df_test_nova.head()

### Do: load all the files with a loop

In [ ]:
# ---------- legacy ----------
legacy_list = []
failed_files = []      # one bad file must not stop the other 100
start = time.time()
for path in legacy_files:
    try:
        df_one = read_legacy_file(path)
        legacy_list.append(df_one)
    except Exception as err:
        failed_files.append((path.name, str(err)[:80]))
df_legacy = pd.concat(legacy_list, ignore_index=True)
print("legacy:", df_legacy.shape, "in", round(time.time() - start), "seconds")

# ---------- nova ----------
nova_list = []
start = time.time()
for path in nova_files:
    try:
        df_one = read_nova_file(path)
        nova_list.append(df_one)
    except Exception as err:
        failed_files.append((path.name, str(err)[:80]))
df_nova = pd.concat(nova_list, ignore_index=True)
print("nova  :", df_nova.shape, "in", round(time.time() - start), "seconds")

check("no file failed to load", len(failed_files) == 0, str(failed_files))

### See: the two tables side by side

In [ ]:
print("LEGACY columns:", list(df_legacy.columns))
print("NOVA columns  :", list(df_nova.columns))
print()
print("columns only in legacy:", set(df_legacy.columns) - set(df_nova.columns))
print("columns only in nova  :", set(df_nova.columns) - set(df_legacy.columns))
print()
print(df_legacy.dtypes)

## 2 · Check every important column on the data as it arrived
We have not cleaned anything yet. Each check says what the column *should* contain. `FAIL` means the column breaks a rule, `WARN` means look at it but it may be fine.

In [ ]:
# turn the date texts into real dates first (the checks need them)
# legacy dates are day first, nova dates are iso with +05:30 at the end (we keep the local time)
df_legacy["slot_dt"] = pd.to_datetime(df_legacy["slot_at"], format="%d/%m/%Y %H:%M", errors="coerce")
df_nova["slot_dt"] = pd.to_datetime(df_nova["slot_at"].str[:19], errors="coerce")
print("legacy slot dates not readable:", df_legacy["slot_dt"].isnull().sum())
print("nova slot dates not readable  :", df_nova["slot_dt"].isnull().sum())

# put the columns we want to check from both systems in one table
check_cols = ["booking_id", "system", "city", "clinic_id", "channel", "source", "patient_id",
              "slot_dt", "test_code", "test_price"]
df_both = pd.concat([df_legacy[check_cols], df_nova[check_cols]], ignore_index=True)
print(df_both.shape)

In [ ]:
# ---------- the list of checks ----------
CITY_CODES = {"Ahmedabad": "AHM", "Bengaluru": "BLR", "Chennai": "CHE",
              "Hyderabad": "HYD", "Jaipur": "JAI", "Pune": "PUN"}

results = []     # one dictionary per check

def add_check(table, column, rule, bad_mask, df, level="FAIL"):
    # bad_mask is True for every row that breaks the rule
    n_bad = int(bad_mask.sum())
    if n_bad == 0:
        status = "PASS"
    else:
        status = level
    examples = ""
    if n_bad > 0:
        examples = ", ".join([str(x) for x in df.loc[bad_mask, column].dropna().unique()[:4]])
    results.append({"table": table, "column": column, "rule": rule, "bad_rows": n_bad,
                    "of_rows": len(df), "status": status, "examples": examples})

d = df_both     # short name
add_check("bookings", "city", "one of the 6 city names", ~d["city"].isin(CITY_CODES), d)
add_check("bookings", "channel", "clinic or home", ~d["channel"].isin(["clinic", "home"]), d)
add_check("bookings", "source", "app, phone or walkin", ~d["source"].isin(["app", "phone", "walkin"]), d)
add_check("bookings", "system", "legacy or nova", ~d["system"].isin(["legacy", "nova"]), d)

# booking id: spaces, pattern, and the city code inside the id
has_spaces = d["booking_id"] != d["booking_id"].str.strip()
add_check("bookings", "booking_id", "no spaces around the id", has_spaces, d)
id_clean = d["booking_id"].str.strip()
ok_legacy = (d["system"] == "legacy") & id_clean.str.match(r"^BK-[A-Z]{3}-\d{6}-\d{6}$")
ok_nova = (d["system"] == "nova") & id_clean.str.match(r"^nova_[a-z]{3}_\d{7}$")
d_ids = d.assign(booking_id=id_clean)
add_check("bookings", "booking_id", "matches the id pattern of its system (after strip)", ~(ok_legacy | ok_nova), d_ids, level="WARN")
code_in_id = id_clean.str.extract(r"^(?:BK-|nova_)([A-Za-z]{3})")[0].str.upper()
add_check("bookings", "booking_id", "city code inside the id agrees with the city column", code_in_id != d["city"].map(CITY_CODES), d_ids)

add_check("bookings", "clinic_id", "pattern like AHM-03", ~d["clinic_id"].astype(str).str.match(r"^[A-Z]{3}-\d{2}$"), d)
add_check("bookings", "clinic_id", "clinic prefix agrees with the city", d["clinic_id"].astype(str).str[:3] != d["city"].map(CITY_CODES), d)
add_check("bookings", "patient_id", "not blank", d["patient_id"].isnull() | (d["patient_id"].astype(str).str.strip() == ""), d)
add_check("bookings", "slot_dt", "a readable date", d["slot_dt"].isnull(), d)
add_check("bookings", "slot_dt", "inside Jan 2025 to Jun 2026", (d["slot_dt"] < "2025-01-01") | (d["slot_dt"] >= "2026-07-01"), d, level="WARN")

price = pd.to_numeric(d["test_price"], errors="coerce")
add_check("bookings", "test_price", "a number above zero where present", (price <= 0) | (price.isnull() & d["test_price"].notnull()), d)
add_check("bookings", "test_price", "not blank", price.isnull(), d, level="WARN")
add_check("bookings", "test_code", "not blank", d["test_code"].isnull(), d, level="WARN")
add_check("bookings", "booking_id", "no row repeated exactly (same booking, test, slot)", d.duplicated(), d, level="WARN")

df_checks = pd.DataFrame(results)
print(df_checks.to_string())

In [ ]:
# every city should have rows in every one of the 18 months
df_both["month"] = df_both["slot_dt"].dt.to_period("M")
table_cm = df_both.pivot_table(index="month", columns="city", values="booking_id", aggfunc="size", fill_value=0)
table_cm = table_cm.loc["2025-01":"2026-06"]
print(table_cm)

n_empty = int((table_cm == 0).sum().sum())
n_cells = table_cm.shape[0] * table_cm.shape[1]
check("every city has rows in every month", n_empty == 0, str(n_empty) + " empty city-months of " + str(n_cells))
print("status_at_export values in nova:", df_nova["status_at_export"].value_counts(dropna=False).to_dict())

# ---------- collections checks ----------
coll_files = sorted((RAW / "collections").glob("*.csv.gz"))
coll_list = []
for path in coll_files:
    coll_list.append(pd.read_csv(path, dtype=str))
df_coll_raw = pd.concat(coll_list, ignore_index=True)
print("collections:", df_coll_raw.shape)

c = df_coll_raw
add_check("collections", "status", "collected or no_show", ~c["status"].isin(["collected", "no_show"]), c)
add_check("collections", "channel", "clinic or home", ~c["channel"].isin(["clinic", "home"]), c)
add_check("collections", "clinic_id", "pattern like AHM-03", ~c["clinic_id"].str.match(r"^[A-Z]{3}-\d{2}$"), c)
add_check("collections", "collected_at", "present when collected", (c["status"] == "collected") & c["collected_at"].isnull(), c)
add_check("collections", "booking_id", "id appears once (double scans)", c["booking_id"].duplicated(), c, level="WARN")
ids_in_bookings = set(id_clean)
add_check("collections", "booking_id", "id exists in bookings (after strip)", ~c["booking_id"].str.strip().isin(ids_in_bookings), c, level="WARN")

df_checks = pd.DataFrame(results)
print(df_checks[df_checks["table"] == "collections"].to_string())
print()
print("PASS:", (df_checks["status"] == "PASS").sum(), " WARN:", (df_checks["status"] == "WARN").sum(), " FAIL:", (df_checks["status"] == "FAIL").sum())
df_checks.to_csv(CLEAN / "column_checks.csv", index=False)

> **Interview angle.** What does a check table like this give you that `df.head()` does not?
>
> *A good answer:* `head()` shows five rows that look fine. A rule applied to every row finds the one row in a thousand that breaks it (a space in an id, a city spelled differently) before it silently drops rows in a join. The table is also the evidence for the panel that we looked before we cleaned.

## 3 · Fix what the checks found (and log each fix)
Only fixes that are safe are done here: spaces, dates, number types and the city spelling. Anything that removes rows waits for notebook 03, where it is logged with its effect on the headline.

In [ ]:
# ---------- spaces around text values ----------
# legacy: every column is text, so strip them all
n_changed = (df_legacy["booking_id"] != df_legacy["booking_id"].str.strip()).sum()
for col in ["booking_id", "city", "clinic_id", "channel", "source", "patient_id", "test_code"]:
    df_legacy[col] = df_legacy[col].str.strip()
print("legacy booking ids that had spaces:", n_changed)

# nova: strip the ids too
for col in ["booking_id", "city", "clinic_id", "patient_id"]:
    df_nova[col] = df_nova[col].str.strip()

# ---------- numbers ----------
for df in [df_legacy, df_nova]:
    df["test_price"] = pd.to_numeric(df["test_price"], errors="coerce")
    df["home_fee"] = pd.to_numeric(df["home_fee"], errors="coerce")

# ---------- created_at as a date ----------
df_legacy["created_dt"] = pd.to_datetime(df_legacy["created_at"], format="%d/%m/%Y %H:%M", errors="coerce")
df_nova["created_dt"] = pd.to_datetime(df_nova["created_at"].str[:19], errors="coerce")

# ---------- city spelling: the plan says Bangalore, our bookings say Bengaluru ----------
for df in [df_legacy, df_nova]:
    print("rows spelled Bangalore:", (df["city"] == "Bangalore").sum())
    df["city"] = df["city"].replace({"Bangalore": "Bengaluru"})

# ---------- legacy has no status columns, so add them empty to be able to stack ----------
df_legacy["status_at_export"] = np.nan
df_legacy["rescheduled_from"] = np.nan

### Do: stack the two systems

In [ ]:
final_cols = ["booking_id", "system", "source_file", "created_dt", "slot_dt", "city", "clinic_id", "channel", "source",
              "patient_id", "test_code", "test_price", "home_fee", "status_at_export", "rescheduled_from"]

df_all = pd.concat([df_legacy[final_cols], df_nova[final_cols]], ignore_index=True)
df_all = df_all.rename(columns={"slot_dt": "slot_at", "created_dt": "created_at"})

print(df_all.shape)
print(df_all.dtypes)
print()
print(df_all.isnull().sum())
df_all.head()

In [ ]:
# the grain trap in numbers: rows per booking is different in the two systems
summary = df_all.groupby("system").agg(rows=("booking_id", "size"), bookings=("booking_id", "nunique"),
                                       first_slot=("slot_at", "min"), last_slot=("slot_at", "max"))
summary["rows_per_booking"] = (summary["rows"] / summary["bookings"]).round(2)
print(summary)
print()
print("rows:", len(df_all), " distinct bookings:", df_all["booking_id"].nunique())

### Check: input = accepted + rejected
A row we cannot place in time or cannot identify is rejected and listed, not deleted silently.

In [ ]:
n_input = len(df_all)

# a row is rejected if it has no booking id or no readable slot date
bad_rows = df_all["booking_id"].isnull() | df_all["slot_at"].isnull()
df_rejects = df_all[bad_rows].copy()
df_rejects["reason"] = "no booking id or no readable slot date"
df_accepted = df_all[~bad_rows].copy()

check("input = accepted + rejected", n_input == len(df_accepted) + len(df_rejects),
      str(n_input) + " = " + str(len(df_accepted)) + " + " + str(len(df_rejects)))
check("every city name is one of the six", df_accepted["city"].isin(CITY_CODES).all())

df_accepted.to_csv(CLEAN / "bookings_all.csv.gz", index=False, compression="gzip")
df_rejects.to_csv(CLEAN / "rejects_bookings.csv", index=False)
print("saved bookings_all.csv.gz:", len(df_accepted), "rows and rejects_bookings.csv:", len(df_rejects), "rows")

## SUM
We now hold every booking of both systems in one table at one grain (one row per test). Record these in `docs/decisions-log.md`:
- the number of repeated header lines dropped, the number of ids with spaces, the Bangalore spelling,
- the PASS / WARN / FAIL table (`column_checks.csv`) and what we decided for every WARN and FAIL,
- why we count from the bookings table (approach-log row 1).

## Before you move on
- What is a booking? Write one sentence. Does a rescheduled nova record count as a second booking?
- How many rows did we reject, and could any of them have been real visits?
- Would a reviewer re-running this on a fresh Codespace get the same counts? Why?